# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

# 1. Fetch token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# 3. The Fix: Explicitly create a Hugging Face secret in DuckDB
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 4. Define the path
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row equals one daily performance record for a specific pseudonymized content item (webpage) for a specific client (`report_date` × `client_hash_id` × `content_hash_id`).
**Time Window:** We are defining our logic using a mid-panel month partition (`month='2026-03'`). The final month of the dataset (June 2026) is strictly excluded and locked away as our natural outcome/test window.
**Tables:** `fact_content_daily_performance` for the metrics, eventually joining with `dim_content` for content metadata.

In [10]:
print("--- Verifying Time Window Bounds ---")
window_query = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{hf_path}'
"""
display(con.execute(window_query).df())

--- Verifying Time Window Bounds ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date
0,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context (IDs for grouping/splitting, never for the model):**
* `client_hash_id`, `content_hash_id`, `report_date`

**Features (Knowable at the decision moment):**
1. `impressions`: Knowable because it is a finalized historical count from past dates.
2. `clicks`: Knowable because it strictly counts past user interactions.
3. `gsc_avg_position`: Knowable because it is computed from historical daily SERP rankings (note: `0` means no data).
4. `content_age_days`: Knowable because the publication date is a fixed event in the past.
5. `ctr`: Knowable because it is a backward-looking ratio derived purely from past impressions and clicks.

**Label / Proxy:**
* `is_declining_label` (The target used to rank pages needing a refresh).

**Excluded (The Deliberate Trap):**
* `trend_direction` and `trend_pct`.
* **Why:** These are computed from the future outcome we are trying to predict. Including them is a leakage trap that would cause the model's accuracy to jump to near 100%, ruining the honest baseline.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
print("--- 1. Proving the Grain (Duplicates check) ---")
# The grain is report_date x client_hash_id x content_hash_id. This should return 0 rows.
grain_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM '{hf_path}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING row_count > 1
LIMIT 5
"""
display(con.execute(grain_query).df())

print("\n--- 2. Proving Row Counts & Availability ---")
# Showing total rows vs rows that actually have GA4 analytics data available
avail_query = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as surviving_ga4_rows
FROM '{hf_path}'
"""
display(con.execute(avail_query).df())

--- 1. Proving the Grain (Duplicates check) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count



--- 2. Proving Row Counts & Availability ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,surviving_ga4_rows
0,9841378,413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Honest Limitations of this Slice:**
History depth differs wildly per client. Rows before a specific client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE`. If we strictly filter for `IS TRUE` to avoid zero-filled rows, we introduce survival bias by blinding the model to early client history or properties that rely purely on Google Search Console. Therefore, a global calendar time window is dangerous; time windows must be evaluated on a per-client basis.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.